오토인코더(Autoencoder)를 활용한 정상치 기반 고장 탐지는 산업계 및 스마트 가전 분야에서 가장 신뢰받는 기술 중 하나입니다.

## 💡 오토인코더의 고장 탐지 원리

오토인코더는 데이터를 압축하는 인코더(Encoder)와 압축된 데이터를 다시 원래대로 복원하는 디코더(Decoder)로 이루어진 신경망입니다. (MakinaRocks)

* 정상 데이터만 학습:
  * 건조기가 정상 작동할 때의 센서 데이터만 모델에 입력하여 "원본 -> 압축 -> 복원" 과정을 거치게 합니다.
  * 모델은 정상 데이터의 패턴을 완벽히 복원하는 방법을 스스로 터득합니다.

* 복원 오차(Reconstruction Error) 확인:
  * 학습이 끝난 모델에 새로운 데이터를 넣고, 복원된 값과 실제 입력값의 차이(오차)를 계산합니다.
  * 정상 데이터가 들어오면 -> 평소에 보던 유형이므로 오차가 매우 작음.
  * 고장 데이터가 들어오면 -> 처음 보는 이상 패턴이므로 복원을 제대로 하지 못해 오차가 매우 큼.

## 정상치 데이터만을 이용해 오토인코더(Autoencoder) 딥러닝 모델로 고장을 탐지

* 오토인코더는 입력 데이터를 압축(Encoding)했다가 다시 원래대로 복원(Decoding)하는 인공신경망입니다.
* 오직 정상 데이터만으로 모델을 학습시키면, 모델은 정상 데이터의 패턴을 매우 잘 복원하게 됩니다.
* 이후 비정상(고장) 데이터가 입력되면 모델이 이를 제대로 복원하지 못해 복원 오차(Reconstruction Error)가 커지며, 이를 통해 고장 유무를 진단합니다.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# 1. 가상의 건조기 복합 센서 데이터 생성
np.random.seed(42)

# [정상 데이터 1,000개] - 오직 이 데이터만 모델 학습(fit)에 사용합니다.
n_normal = 1000
t_train = np.linspace(0, 20 * np.pi, n_normal)
train_temp = 50 + 20 * np.sin(t_train) + np.random.normal(0, 1.0, n_normal)
train_humid = 55 + 35 * np.cos(t_train) + np.random.normal(0, 1.0, n_normal)
train_vib = 0.3 + 0.1 * np.sin(t_train * 2) + np.random.normal(0, 0.01, n_normal)
train_curr = 8 + 3 * np.cos(t_train / 2) + np.random.normal(0, 0.1, n_normal)
X_train_normal = np.column_stack([train_temp, train_humid, train_vib, train_curr])

# [테스트용 정상 데이터 200개]
n_test_normal = 200
t_test = np.linspace(20 * np.pi, 24 * np.pi, n_test_normal)
test_norm_temp = 50 + 20 * np.sin(t_test) + np.random.normal(0, 1.0, n_test_normal)
test_norm_humid = 55 + 35 * np.cos(t_test) + np.random.normal(0, 1.0, n_test_normal)
test_norm_vib = 0.3 + 0.1 * np.sin(t_test * 2) + np.random.normal(0, 0.01, n_test_normal)
test_norm_curr = 8 + 3 * np.cos(t_test / 2) + np.random.normal(0, 0.1, n_test_normal)
X_test_normal = np.column_stack([test_norm_temp, test_norm_humid, test_norm_vib, test_norm_curr])

# [테스트용 고장 데이터 50개] (온도 과열 및 비정상 전류/진동 발생)
n_fault = 50
fault_temp = np.random.uniform(75, 85, n_fault)
fault_humid = np.random.uniform(60, 70, n_fault)
fault_vib = np.random.uniform(0.4, 0.5, n_fault)
fault_curr = np.random.uniform(11, 13, n_fault)
X_test_fault = np.column_stack([fault_temp, fault_humid, fault_vib, fault_curr])


# 2. 데이터 전처리 (Min-Max Scaling)
# 오토인코더 출력층의 sigmoid 활성화 함수를 위해 데이터를 0~1 사이로 압축합니다.
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_normal) # 정상 데이터로만 기준 설정
X_test_norm_scaled = scaler.transform(X_test_normal)
X_test_fault_scaled = scaler.transform(X_test_fault)


# 3. Autoencoder 모델 아키텍처 구축
input_dim = X_train_scaled.shape[1] # 입력 차원: 4 (온도, 습도, 진동, 전류)

# 인코더 레이어 (4차원 -> 2차원 압축)
input_layer = Input(shape=(input_dim,))
encoded = Dense(2, activation='relu')(input_layer)

# 디코더 레이어 (2차원 -> 4차원 복원)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

# 전체 오토인코더 모델 선언 및 컴파일
autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse') # 손실함수로 평균제곱오차(MSE) 사용


# 4. 모델 학습 (정상 데이터로만 진행)
# 입력(X)과 타깃(y)이 모두 자기 자신인 `X_train_scaled`인 것이 핵심입니다.
autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    verbose=0 # 학습 과정 출력 생략
)


# 5. 고장 진단 임계치(Threshold) 설정
# 학습에 쓰인 정상 데이터의 복원 오차를 구합니다.
train_predictions = autoencoder.predict(X_train_scaled, verbose=0)
mse_train = np.mean(np.power(X_train_scaled - train_predictions, 2), axis=1)

# 정상 복원 오차의 '평균 + 3 * 표준편차'를 임계치로 지정합니다.
threshold = np.mean(mse_train) + 3 * np.std(mse_train)


# 6. 테스트 데이터를 통한 고장 진단
# 정상 테스트 데이터 복원 오차 계산
test_norm_preds = autoencoder.predict(X_test_norm_scaled, verbose=0)
mse_norm = np.mean(np.power(X_test_norm_scaled - test_norm_preds, 2), axis=1)

# 고장 테스트 데이터 복원 오차 계산
test_fault_preds = autoencoder.predict(X_test_fault_scaled, verbose=0)
mse_fault = np.mean(np.power(X_test_fault_scaled - test_fault_preds, 2), axis=1)

# 진단 판정 (임계치를 넘으면 고장(-1), 안 넘으면 정상(1))
pred_norm = np.where(mse_norm > threshold, -1, 1)
pred_fault = np.where(mse_fault > threshold, -1, 1)


# 7. 결과 시각화
plt.figure(figsize=(10, 5))

# 오차 시각화
plt.plot(mse_norm, label='Test Normal Error', color='blue', alpha=0.7)
plt.plot(np.arange(len(mse_norm), len(mse_norm) + len(mse_fault)), 
         mse_fault, label='Test Fault Error', color='red', alpha=0.7)

plt.axhline(y=threshold, color='green', linestyle='--', label=f'Threshold ({threshold:.4f})')
plt.title('Fault Detection via Autoencoder Reconstruction Error')
plt.ylabel('Reconstruction Error (MSE)')
plt.xlabel('Data Index')
plt.legend()
plt.grid(True)
plt.show()

# 8. 탐지율 콘솔 출력
print(f"[진단 결과]")
print(f"임계치(Threshold): {threshold:.5f}")
print(f"정상 데이터 {n_test_normal}개 중 탐지한 정상 개수: {np.sum(pred_norm == 1)}개")
print(f"고장 데이터 {n_fault}개 중 탐지한 고장 개수: {np.sum(pred_fault == -1)}개")

#### 🔍 코드의 핵심 포인트

* 복원 오차(Reconstruction Error)의 활용:
  * 이 모델은 데이터 분포의 기하학적 형태를 그대로 연산하는 LLE와 달리, '학습하지 않은 비정상 데이터를 제대로 복원하지 못해 에러가 커지는 현상'을 이용합니다.

* 임계치(Threshold) 설정:
  * 평균 + (3 * 표준편차) 방식은 99.7%의 정상 데이터 범위를 감싸 안는 통계적 방법입니다.
  * 현장 환경의 노이즈 크기에 따라 2.5 혹은 3.5 등으로 이 숫자를 조정하며 정확도를 맞춥니다.

* 학습 입력값:
  * autoencoder.fit(X_train_scaled, X_train_scaled, ...)를 통해 레이블 없이 오직 정상 피처만으로 비지도 학습을 수행합니다.

### 💻 Autoencoder 기반 건조기 고장 진단 코드 요약

* 제공된 코드 예제는 가상의 4개 센서 데이터(온도, 습도, 진동, 전류)를 생성하여 오토인코더(Autoencoder) 모델을 통해 정상 데이터(X_train_normal)만 학습하고, 테스트 데이터에서 정상과 고장 데이터를 구분하는 과정을 보여줍니다.
* sklearn.preprocessing.MinMaxScaler로 스케일링한 후, 2차원 잠재 공간으로 압축(Dense(2, activation='relu'))하는 인코더와 원래 차원으로 복원(Dense(input_dim, activation='sigmoid'))하는 디코더를 연결하여 TensorFlow/Keras 모델을 구현합니다.
* MSE(Mean Squared Error) 기반의 복원 오차를 계산하고, 훈련 데이터의 '평균 + 3*표준편차'를 임계치(Threshold)로 설정하여 딥러닝 기반 이상 탐지를 수행합니다.

### 💡 실무 적용 시 Autoencoder가 LLE(Locally Linear Embedding)보다 유리한 점

* 실시간 처리 속도:
  * LLE는 새로운 데이터가 들어올 때마다 기존 데이터와의 거리를 재계산해야 하므로 데이터 양에 따라 계산 부하가 급증합니다.
  * 반면 Autoencoder는 이미 학습된 신경망(가중치)에 데이터만 통과시키면 되어 실시간 모니터링에 훨씬 효율적입니다.
* 직관적인 고장 진단:
  * 데이터가 복잡한 공간에서 얼마나 떨어져 있는지 계산하는 대신, '학습된 정상 패턴과 얼마나 다른지'를 복원 오차(Reconstruction Error)라는 수치로 명확히 보여줍니다.

## 복원오차 & Threshold & 이상 탐지 수행 및 판전

### 1. 복원 오차(Reconstruction Error) 계산

학습이 끝난 모델에 데이터를 입력하여 원래 값과의 차이(오차)를 구합니다.

In [ ]:
# Train 데이터(정상 데이터)에 대한 예측값 생성
X_train_pred = model.predict(X_train)

# 각 샘플별 평균 절대 오차(MAE) 계산
train_mae_loss = np.mean(np.abs(X_train_pred - X_train), axis=(1, 2))

### 2. 가장 합리적인 Threshold(임계값)를 찾는 3가지 방법

임계값을 결정할 때는 목적과 보유한 라벨 데이터(정상/이상 여부) 유무에 따라 아래의 3가지 공학적 방법 중 하나를 선택하는 것이 가장 합리적입니다.

_*방법 ①: 정규분포(가우시안) 기반 통계적 설정 (라벨이 없을 때 추천)*_

- 정상 데이터의 복원 오차가 정규분포를 따른다고 가정하고, $[평균 + K * 표준편차]$ 공식을 사용합니다. 
- 산업계에서는 일반적으로 $k=3$ (3-Sigma 원칙)을 사용하여 99.7%의 정상 범위를 커버하고 이를 벗어나는 0.3%를 이상으로 간주합니다.

In [ ]:
mean_loss = np.mean(train_mae_loss)
std_loss = np.std(train_mae_loss)

# k=3 설정 (상황에 따라 2.5 ~ 4 사이 조정)
threshold_stat = mean_loss + (3 * std_loss)
print(f"통계적 기준 임계값: {threshold_stat}")

__*방법 ②: Precision-Recall Curve 및 F1-Score 최대화 (라벨이 있을 때 최선)*__

과거 고장 이력 등 '이상 데이터'에 대한 라벨(True Label)이 소량이라도 존재한다면, 불균형 데이터셋 탐지에 가장 효과적인 __F1-Score를 최대로 만드는 지점을__ 임계값으로 설정하는 것이 가장 합리적입니다.

In [ ]:
from sklearn.metrics import precision_recall_curve

# test_mae_loss: 이상이 포함된 테스트 데이터의 복원 오차
# y_test: 실제 이상 여부 라벨 (0: 정상, 1: 이상)
precisions, recalls, thresholds = precision_recall_curve(y_test, test_mae_loss)

# F1-Score 계산
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# F1-Score가 최대가 되는 인덱스의 임계값 선택
best_idx = np.argmax(f1_scores)
threshold_f1 = thresholds[best_idx]
print(f"최적의 F1-Score 기준 임계값: {threshold_f1}")

__*방법 ③: 백분위수(Percentile) 기준 설정 (현장 도메인 지식 반영)*__

공정 전문가의 경험을 토대로 "우리 건조기 설비는 가동 시간 중 약 __1%__ 정도만 이상 징후를 보인다"라는 도메인 지식이 있다면 백분위수를 활용해 임계값을 고정합니다.

In [ ]:
# 상위 1%를 이상으로 간주할 때 (99% 분위수 설정)
threshold_percentile = np.percentile(train_mae_loss, 99)
print(f"상위 1% 기준 임계값: {threshold_percentile}")

### 3. 이상 탐지 수행 및 판정
결정된 threshold를 기반으로 새로운 데이터에 대해 이상 유무를 판정합니다.

In [ ]:
# 최종 선택한 threshold 대입
threshold = threshold_f1  # 또는 threshold_stat

# 이상 판정 (True: 이상 발생, False: 정상)
is_anomaly = test_mae_loss > threshold

건조기 이상 탐지 데이터를 직접 보유하고 계신다면 테스트 데이터에 정상/이상 라벨이 지정되어 있는지 알려주세요. 그에 맞춰 F1-score 최적화 등 코드를 직접 커스텀해 드리겠습니다.